## Establecer conexión con SQL

In [1]:
import mysql.connector
from mysql.connector import Error
import pandas as pd

In [2]:
try:
    connection = mysql.connector.connect(host='212.227.90.6',
                                         database='Equip_15',
                                         user='Equipo15',
                                         password = 'E1q2u3i4p5o15')
    if connection.is_connected():
        db_Info = connection.get_server_info()
        print("Connected to MySQL Server version ", db_Info)
        RRHH = pd.read_sql(f"SELECT * FROM RRHH_15092025", con=connection)
        
       
except Error as e:
    print("Error while connecting to MySQL", e)

Connected to MySQL Server version  8.0.43-0ubuntu0.24.04.1


C:\Users\gemma\AppData\Local\Temp\ipykernel_3352\859305398.py:7: DeprecationWarning: Call to deprecated function get_server_info. Reason: 
    The property counterpart 'server_info' should be used instead.

  db_Info = connection.get_server_info()
C:\Users\gemma\AppData\Local\Temp\ipykernel_3352\859305398.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  RRHH = pd.read_sql(f"SELECT * FROM RRHH_15092025", con=connection)


In [3]:
pd.set_option("display.max.columns", None)
#RRHH.describe() #variables numèriques
#RRHH.describe(include = 'object') #variables categòriques
RRHH.dtypes

ID                          int64
Reason_absence              int64
Month_absence               int64
Day_week                    int64
Seasons                     int64
Transportation_expense      int64
Distance_Residence_Work     int64
Service_time                int64
Age                         int64
Work_load_Average_day      object
Hit_target                  int64
Disciplinary_failure       object
Education                  object
Son                        object
Social_drinker             object
Social_smoker              object
Pet                        object
Weight                      int64
Height                      int64
Body_mass_index             int64
Absenteeism_hours           int64
dtype: object

# Limpieza de datos

## Cambiar los tipos de datos

In [4]:
# Change from numeric to categorical
for col in ["ID", "Reason_absence","Month_absence","Day_week","Seasons"]:
    RRHH[col] = RRHH[col].astype("object")
    
# Change from object to numeric
for col in ["Son", "Pet"]:
    RRHH[col] = RRHH[col].astype("int")

# Change decimal separator and change type to float
RRHH["Work_load_Average_day"] = (
    RRHH["Work_load_Average_day"]
    .str.replace(",", ".", regex=False) 
    .astype(float)
)

RRHH.dtypes

ID                          object
Reason_absence              object
Month_absence               object
Day_week                    object
Seasons                     object
Transportation_expense       int64
Distance_Residence_Work      int64
Service_time                 int64
Age                          int64
Work_load_Average_day      float64
Hit_target                   int64
Disciplinary_failure        object
Education                   object
Son                          int64
Social_drinker              object
Social_smoker               object
Pet                          int64
Weight                       int64
Height                       int64
Body_mass_index              int64
Absenteeism_hours            int64
dtype: object

In [5]:
RRHH.describe()
#RRHH.describe(include = 'object') 
#RRHH.describe(include = 'category') 

,Transportation_expense,Distance_Residence_Work,Service_time,Age,Work_load_Average_day,Hit_target,Son,Pet,Weight,Height,Body_mass_index,Absenteeism_hours
count,740.000000,740.000000,740.000000,740.000000,740.000000,740.000000,740.000000,740.000000,740.000000,740.000000,740.000000,740.000000
mean,221.329730,29.631081,12.554054,36.450000,271.490235,94.587838,1.018919,0.745946,79.035135,172.114865,26.677027,6.924324
std,66.952223,14.836788,4.384873,6.478772,39.058116,3.779313,1.098489,1.318258,12.883211,6.034995,4.285452,13.330998
min,118.000000,5.000000,1.000000,27.000000,205.917000,81.000000,0.000000,0.000000,56.000000,163.000000,19.000000,0.000000
25%,179.000000,16.000000,9.000000,31.000000,244.387000,93.000000,0.000000,0.000000,69.000000,169.000000,24.000000,2.000000
50%,225.000000,26.000000,13.000000,37.000000,264.249000,95.000000,1.000000,0.000000,83.000000,170.000000,25.000000,3.000000
75%,260.000000,50.000000,16.000000,40.000000,294.217000,97.000000,2.000000,1.000000,89.000000,172.000000,31.000000,8.000000
max,388.000000,52.000000,29.000000,58.000000,378.884000,100.000000,4.000000,8.000000,108.000000,196.000000,38.000000,120.000000


## Eliminar duplicados exactos

In [6]:
RRHH = RRHH.drop_duplicates()
RRHH.shape[0]

706

# Transformación

## Renombrar niveles variables categóricas

In [7]:
# MESES
## Dejamos la columna con número para poderlo ordenar en PowerBI
RRHH["Month_absence_order"] = RRHH["Month_absence"]

# Especificamos el nombre del mes
month_map = {
    1: "Enero", 2: "Febrero", 3: "Marzo", 4: "Abril",
    5: "Mayo", 6: "Junio", 7: "Julio", 8: "Agosto",
    9: "Septiembre", 10: "Octubre", 11: "Noviembre", 12: "Diciembre"
}

RRHH["Month_absence"] = RRHH["Month_absence"].replace(month_map)


# DÍAS DE LA SEMANA
## Dejamos la columna con número para poderlo ordenar en PowerBI
RRHH["Day_week_order"] = RRHH["Day_week"]

# Especificamos el día de la semana
day_map = {
    2: "Lunes", 3: "Martes", 4: "Miercoles", 5: "Jueves", 6: "Viernes"
}
RRHH["Day_week"] = RRHH["Day_week"].replace(day_map)

# ESTACIONES DEL AÑO
spanish_season_map = {
    1: "Invierno", 2: "Otono", 3: "Verano", 4: "Primavera"
}
RRHH["Seasons"] = RRHH["Seasons"].replace(spanish_season_map)

# EDUCACIÓN
# Especificamos la educación
RRHH["Education"] = RRHH["Education"].astype(str).str.strip()

education_map = {
    "1": "High school",
    "2": "Graduate",
    "3": "Postgraduate",
    "4": "Master/Doctor"
}

RRHH["Education"] = RRHH["Education"].replace(education_map)


## Exportar a csv

In [8]:
RRHH.to_csv("RRHH_150925_clean.csv", index=False, sep=",")

## Separamos la información en dos tablas 

### Hechos

In [9]:
RRHH_hechos = RRHH
RRHH_hechos.head()

,ID,Reason_absence,Month_absence,Day_week,Seasons,Transportation_expense,Distance_Residence_Work,Service_time,Age,Work_load_Average_day,Hit_target,Disciplinary_failure,Education,Son,Social_drinker,Social_smoker,Pet,Weight,Height,Body_mass_index,Absenteeism_hours,Month_absence_order,Day_week_order
0,14,11,Noviembre,Lunes,Primavera,155,12,14,34,284.031,97,0,High school,2,1,0,0,95,196,25,120,11,2
1,36,13,Abril,Miercoles,Verano,118,13,18,50,239.409,98,0,High school,1,1,0,0,98,178,31,120,4,4
2,9,6,Julio,Martes,Invierno,228,14,16,58,264.604,93,0,High school,2,0,0,1,65,172,22,120,7,3
3,28,9,Julio,Martes,Invierno,225,26,9,28,230.290,92,0,High school,1,0,0,2,69,169,24,112,7,3
4,9,12,Marzo,Martes,Otono,228,14,16,58,222.196,99,0,High school,2,0,0,1,65,172,22,112,3,3


In [10]:
# Seleccionamos las columnas de interés

RRHH_hechos = RRHH[["ID", "Reason_absence", "Month_absence","Day_week","Seasons","Work_load_Average_day","Hit_target","Disciplinary_failure","Absenteeism_hours","Month_absence_order","Day_week_order"]]

### Dimensiones

In [11]:
RRHH_dimensiones = RRHH[["ID", "Transportation_expense", "Distance_Residence_Work","Service_time","Age","Education","Son","Social_drinker","Social_smoker","Pet","Weight","Height","Body_mass_index"]]

In [12]:
RRHH_dimensiones = RRHH_dimensiones.drop_duplicates(subset=["ID"], keep="last").reset_index(drop=True)
RRHH_dimensiones.shape[0]
RRHH_dimensiones.head()

,ID,Transportation_expense,Distance_Residence_Work,Service_time,Age,Education,Son,Social_drinker,Social_smoker,Pet,Weight,Height,Body_mass_index
0,6,189,29,13,33,High school,2,0,0,2,69,167,25
1,16,118,15,24,46,High school,2,1,1,0,75,175,25
2,25,235,16,8,32,Postgraduate,0,0,0,0,75,178,25
3,12,233,51,1,31,Graduate,1,1,0,8,68,178,21
4,27,184,42,7,27,High school,0,0,0,0,58,167,21


In [13]:
RRHH_hechos.to_csv("RRHH_150925_hechos.csv", index=False, sep=",")
RRHH_dimensiones.to_csv("RRHH_150925_dimensiones.csv", index=False, sep=",")

# Clustering

In [14]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Agrupar por ids
socio = RRHH.groupby("ID").first()[[
    "Age", "Distance_Residence_Work", "Transportation_expense",
    "Son", "Education", "Weight", "Height", "Pet", 
    "Social_drinker", "Social_smoker"
]]
socio["Social_drinker"] = socio["Social_drinker"].astype(int)
socio["Social_smoker"] = socio["Social_smoker"].astype(int)

# Education sí es categórica, el resto no
socio["Education"] = socio["Education"].astype(str).fillna("Desconocido")

# Crear dummies SOLO para education
df_socio = pd.get_dummies(socio, columns=["Education"], drop_first=True)

# Escalado
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_socio)

# KMeans clustering
kmeans = KMeans(n_clusters=4, random_state=42)
clusters = kmeans.fit_predict(X_scaled)
df_socio["cluster"] = clusters

# Resumen por cluster
cluster_summary = df_socio.groupby("cluster").mean()
print(cluster_summary)

# Ids de cada grupo
for c in sorted(df_socio["cluster"].unique()):
    ids = df_socio[df_socio["cluster"] == c].index.tolist()
    print(f"\nCluster {c} ({len(ids)} empleados):")
    print(ids)

               Age  Distance_Residence_Work  Transportation_expense       Son  \
cluster                                                                         
0        34.666667                18.000000              179.000000  1.333333   
1        40.888889                30.222222              221.666667  0.833333   
2        38.000000                30.100000              276.800000  2.100000   
3        32.000000                16.000000              249.400000  0.200000   

            Weight      Height       Pet  Social_drinker  Social_smoker  \
cluster                                                                   
0        88.000000  187.666667  0.666667             1.0       0.333333   
1        80.833333  171.111111  1.555556             0.5       0.000000   
2        74.100000  170.600000  1.500000             0.7       0.600000   
3        76.400000  176.200000  0.200000             0.0       0.000000   

         Education_High school  Education_Master/Doctor  \
clu

## Clustering 2

K-means només amb variables numèriques

In [15]:
# K-means assumeix que les variables són numèriques i que té sentit calcular mitjanes i distàncies euclidianes i això no funciona bé amb variables categòriques. 
# Per tant, eliminem les categòriques d'aquest clustering a veure què passa. 

# També traiem Weight i Height perquè no crec que tinguin massa sentit. 
# Agrupar por ids
df_socio_numericas = RRHH.groupby("ID").first()[[
    "Age", "Distance_Residence_Work", "Transportation_expense",
    "Son", "Pet"
]]


# Escalado
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_socio_numericas)

# KMeans clustering
kmeans = KMeans(n_clusters=4, random_state=42)
clusters = kmeans.fit_predict(X_scaled)
df_socio_numericas["cluster"] = clusters

# Resumen por cluster
cluster_summary = df_socio_numericas.groupby("cluster").mean()
print(cluster_summary)

# Ids de cada grupo
for c in sorted(df_socio_numericas["cluster"].unique()):
    ids = df_socio_numericas[df_socio_numericas["cluster"] == c].index.tolist()
    print(f"\nCluster {c} ({len(ids)} empleados):")
    print(ids)

               Age  Distance_Residence_Work  Transportation_expense       Son  \
cluster                                                                         
0        41.352941                22.941176              225.705882  1.941176   
1        32.272727                27.272727              219.272727  0.090909   
2        39.666667                31.333333              195.333333  1.000000   
3        40.600000                39.000000              341.400000  0.800000   

              Pet  
cluster            
0        0.705882  
1        0.181818  
2        7.000000  
3        2.200000  

Cluster 0 (17 empleados):
[1, 5, 6, 7, 8, 9, 11, 13, 14, 16, 17, 20, 26, 29, 33, 35, 36]

Cluster 1 (11 empleados):
[3, 18, 19, 21, 22, 24, 25, 27, 28, 30, 34]

Cluster 2 (3 empleados):
[2, 4, 12]

Cluster 3 (5 empleados):
[10, 15, 23, 31, 32]


# Clustering mixte

Amb pes i altura

In [ ]:
from kmodes.kprototypes import KPrototypes

# Agrupar por ID 
df_orig = RRHH.groupby("ID").first()[[
    "Age", "Distance_Residence_Work", "Transportation_expense",
    "Son", "Pet", "Weight", "Height", "Education", "Social_drinker", "Social_smoker"
]].copy()

# Columnas numéricas y categóricas
num_cols = ["Age", "Distance_Residence_Work", "Transportation_expense", "Son", "Pet", "Weight", "Height"]
cat_cols = ["Education", "Social_drinker", "Social_smoker"]

# Preparamos un dataframe para K-Prototypes: escalamos SOLO las numéricas
df_kp = df_orig.copy()
scaler = StandardScaler()
df_kp[num_cols] = scaler.fit_transform(df_kp[num_cols])

# Aseguramos que las categóricas sean strings (requerido por k-prototypes)
for c in cat_cols:
    df_kp[c] = df_kp[c].astype(str)
# índices de columnas categóricas para k-prototypes
cat_idx = [df_kp.columns.get_loc(c) for c in cat_cols]

# Ejecutar K-Prototypes
kproto = KPrototypes(n_clusters=4, random_state=42, init='Cao')
clusters = kproto.fit_predict(df_kp.values, categorical=cat_idx)
df_orig["cluster"] = clusters

# Resumen numérico en unidades originales (medias)
cluster_num_summary = df_orig.groupby("cluster")[num_cols].mean()
print("Resumen numérico (unidades originales):")
print(cluster_num_summary.round(3))

# Resumen categórico: proporciones por cluster (por ejemplo % bebedores/fumadores y distribución de Education)
print("\nProporciones por cluster (categóricas):")
for c in cat_cols:
    prop = df_orig.groupby("cluster")[c].value_counts(normalize=True).unstack(fill_value=0)
    print(f"\n== {c} ==")
    print((prop*100).round(1))   # en porcentaje

# Modo (valor más frecuente) por cluster para las categóricas
mode_cat = df_orig.groupby("cluster")[cat_cols].agg(lambda x: x.mode().iloc[0] if not x.mode().empty else None)
print("\nModo (valor más frecuente) de las categóricas por cluster:")
print(mode_cat)

# Tamaño e IDs por cluster
print("\nTamaño por cluster:")
print(df_orig['cluster'].value_counts().sort_index())
print("\nIDs por cluster:")
for c in sorted(df_orig['cluster'].unique()):
    ids = df_orig[df_orig['cluster'] == c].index.tolist()
    print(f"\nCluster {c} ({len(ids)} empleados):")
    print(ids)


Resumen numérico (unidades originales):
            Age  Distance_Residence_Work  Transportation_expense    Son  \
cluster                                                                   
0        39.000                   40.000                 235.143  0.857   
1        36.000                   30.500                 274.667  1.750   
2        47.222                   19.222                 219.333  1.333   
3        31.250                   20.000                 203.250  0.250   

           Pet  Weight   Height  
cluster                          
0        4.143  90.143  170.714  
1        0.917  70.083  170.167  
2        0.667  85.000  173.778  
3        0.000  75.625  178.625  

Proporciones por cluster (categóricas):

== Education ==
Education  Graduate  High school  Master/Doctor  Postgraduate
cluster                                                      
0              14.3         85.7            0.0           0.0
1               8.3         91.7            0.0           0.0

# Clustering mixte (CLUSTERING UTILITZAT FINAL)

Amb IMC

In [18]:
# Agrupar por ID 
df_orig = RRHH.groupby("ID").first()[[
    "Age", "Distance_Residence_Work", "Transportation_expense",
    "Son", "Pet", "Body_mass_index", "Education", "Social_drinker", "Social_smoker"
]].copy()

# Columnas numéricas y categóricas
num_cols = ["Age", "Distance_Residence_Work", "Transportation_expense", "Son", "Pet", "Body_mass_index"]
cat_cols = ["Education", "Social_drinker", "Social_smoker"]

# Preparamos un dataframe para K-Prototypes: escalamos SOLO las numéricas
df_kp = df_orig.copy()
scaler = StandardScaler()
df_kp[num_cols] = scaler.fit_transform(df_kp[num_cols])

# Aseguramos que las categóricas sean strings (requerido por k-prototypes)
for c in cat_cols:
    df_kp[c] = df_kp[c].astype(str)
# índices de columnas categóricas para k-prototypes
cat_idx = [df_kp.columns.get_loc(c) for c in cat_cols]

# Ejecutar K-Prototypes
kproto = KPrototypes(n_clusters=4, random_state=42, init='Cao')
clusters = kproto.fit_predict(df_kp.values, categorical=cat_idx)
df_orig["cluster"] = clusters

# Resumen numérico en unidades originales (medias)
cluster_num_summary = df_orig.groupby("cluster")[num_cols].mean()
print("Resumen numérico (unidades originales):")
print(cluster_num_summary.round(3))

# Resumen categórico: proporciones por cluster (por ejemplo % bebedores/fumadores y distribución de Education)
print("\nProporciones por cluster (categóricas):")
for c in cat_cols:
    prop = df_orig.groupby("cluster")[c].value_counts(normalize=True).unstack(fill_value=0)
    print(f"\n== {c} ==")
    print((prop*100).round(1))   # en porcentaje

# Modo (valor más frecuente) por cluster para las categóricas
mode_cat = df_orig.groupby("cluster")[cat_cols].agg(lambda x: x.mode().iloc[0] if not x.mode().empty else None)
print("\nModo (valor más frecuente) de las categóricas por cluster:")
print(mode_cat)

# Tamaño e IDs por cluster
print("\nTamaño por cluster:")
print(df_orig['cluster'].value_counts().sort_index())
print("\nIDs por cluster:")
for c in sorted(df_orig['cluster'].unique()):
    ids = df_orig[df_orig['cluster'] == c].index.tolist()
    print(f"\nCluster {c} ({len(ids)} empleados):")
    print(ids)

Resumen numérico (unidades originales):
            Age  Distance_Residence_Work  Transportation_expense   Son    Pet  \
cluster                                                                         
0        45.222                   31.111                 203.556  1.00  2.111   
1        32.800                   47.600                 304.200  2.00  3.400   
2        31.700                   24.900                 223.300  0.10  0.200   
3        41.000                   17.667                 246.333  1.75  0.667   

         Body_mass_index  
cluster                   
0                 32.778  
1                 24.400  
2                 23.500  
3                 24.917  

Proporciones por cluster (categóricas):

== Education ==
Education  Graduate  High school  Master/Doctor  Postgraduate
cluster                                                      
0               0.0        100.0            0.0           0.0
1              20.0         80.0            0.0           0.0
2    

## Cerrar conexión

In [ ]:
connection.close()
